In [2]:
import numpy as np
import pandas as pd
import pywt
from scipy.stats import entropy


In [3]:

def compute_wavelet_features(window, wavelet='db4', level=3):
    """Compute Wavelet Transform features for all EEG channels in a given window."""
    wavelet_features = []
    
    for i in range(window.shape[1]):  # Iterate over EEG channels
        signal = window[:, i]

        # Apply Discrete Wavelet Transform (DWT)
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        
        # Extract Approximation and Detail Coefficients
        approx_coeffs = coeffs[0]  # Approximation (Low-frequency)
        detail_coeffs = np.concatenate(coeffs[1:])  # All Detail (High-frequency)

        # Compute Features
        channel_features = [
            np.mean(approx_coeffs),  # Approximation Mean
            np.std(approx_coeffs),   # Approximation Std Dev
            np.sum(approx_coeffs ** 2),  # Approximation Energy
            entropy(np.abs(approx_coeffs) + 1e-10),  # Approximation Entropy
            np.mean(detail_coeffs),  # Detail Mean
            np.std(detail_coeffs),   # Detail Std Dev
            np.sum(detail_coeffs ** 2),  # Detail Energy
            entropy(np.abs(detail_coeffs) + 1e-10)   # Detail Entropy
        ]
        
        wavelet_features.extend(channel_features)  # Flatten features into a single list
    
    return wavelet_features


In [4]:

def sliding_window_wavelet_features(eeg_data, outcomes, eeg_columns, window_size, step_size, wavelet='db4', level=3):
    """Extract Wavelet Transform features using a sliding window approach."""
    all_wavelet_features = []
    targets = []
    n_samples = eeg_data.shape[0]
    
    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        window = eeg_data[start:end]
        outcome_window = outcomes[start:end]

        # Compute Wavelet features for this window
        wavelet_values = compute_wavelet_features(window, wavelet=wavelet, level=level)

        all_wavelet_features.append(wavelet_values)
        targets.append(1 if np.any(outcome_window) else 0)  # Assign target based on Outcome
    
    return np.array(all_wavelet_features), np.array(targets)


In [5]:

# Main processing
if __name__ == "__main__":
    # Load EEG data
    eeg_data_path = '/Users/puchku-home/Study/PROJECT/EEG/EEG Assets/chbmit_preprocessed_data.csv'
    data = pd.read_csv(eeg_data_path)
    eeg_columns = [col for col in data.columns if col != 'Outcome']

    # Convert to NumPy arrays
    eeg_data = np.asarray(data[eeg_columns].values, dtype=np.float32)
    outcomes = np.asarray(data['Outcome'].values, dtype=np.float32)

    # Windowing parameters
    fs = 256  # Sampling frequency (not directly used here but relevant for context)
    window_size = fs * 1  # 1-second windows
    step_size = window_size // 2  # 50% overlap

    # Compute Wavelet Transform features
    wavelet_features, targets = sliding_window_wavelet_features(
        eeg_data, outcomes, eeg_columns, window_size, step_size, wavelet='db4', level=3)

    # Generate feature names dynamically based on the number of coefficients per channel
    feature_names = []
    for col in eeg_columns:
        feature_names.extend([
            f"{col}_approx_mean", f"{col}_approx_std", f"{col}_approx_energy", f"{col}_approx_entropy",
            f"{col}_detail_mean", f"{col}_detail_std", f"{col}_detail_energy", f"{col}_detail_entropy"
        ])
    
    wavelet_df = pd.DataFrame(wavelet_features, columns=feature_names)
    wavelet_df['target'] = targets

    output_file_path = '/Users/puchku-home/Downloads/Frequency Feature  Generalised/Wavelet_Features.csv'
    wavelet_df.to_csv(output_file_path, index=False)
    
    print(f"✅ Wavelet Transform feature extraction complete. Features saved to '{output_file_path}'")


✅ Wavelet Transform feature extraction complete. Features saved to '/Users/puchku-home/Downloads/Frequency Feature  Generalised/Wavelet_Features.csv'
